<a href="https://colab.research.google.com/github/Arjx01/ML2_Arjun-M_4NI23CI019/blob/main/ML_2_5a.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
import pandas as pd
import math

In [28]:
def foil_gain(p0, n0, p1, n1):
    """
    FOIL Gain:

    Gain = p1 * (log2(p1/(p1+n1)) - log2(p0/(p0+n0)))

    p0 = positive examples before adding literal
    n0 = negative examples before adding literal
    p1 = positive examples after adding literal
    n1 = negative examples after adding literal
    """

    if p1 == 0:
        return float("-inf")
    if p0 + n0 == 0:
        return float("-inf")
    if p1 + n1 == 0:
        return float("-inf")
    old_probability = p0 / (p0 + n0)
    new_probability = p1 / (p1 + n1)
    if old_probability == 0 or new_probability == 0:
        return float("-inf")
    return p1 * (
        math.log2(new_probability)
        - math.log2(old_probability)
    )

In [30]:
def satisfies(example, rule):
    """
    Check whether an example satisfies all literals in a rule.

    Example rule:
        [
            ("Weather", "Overcast"),
            ("Wind", "Weak")
        ]
    """

    for attribute, value in rule:
        if str(example[attribute]) != str(value):
            return False
    return True

In [31]:
def covered_examples(df, rule):
    """
    Return examples covered by the current rule.
    """

    if len(rule) == 0:
        return df.copy()
    mask = df.apply(
        lambda row: satisfies(row, rule),
        axis=1
    )
    return df[mask].copy()

In [32]:
def generate_candidate_literals(df, rule, features):
    """
    Generate possible attribute=value literals.

    Do not reuse an attribute already present in the rule.
    """

    used_attributes = {attribute for attribute, value in rule}
    candidates = []
    for feature in features:
        if feature in used_attributes:
            continue
        values = df[feature].dropna().unique()
        for value in values:
            candidates.append((feature, value))
    return candidates

In [33]:
def literal_to_string(literal):
    attribute, value = literal
    return f"{attribute} = {value}"

In [34]:
def rule_to_string(rule):
    """
    Convert a rule to readable form.
    """
    if len(rule) == 0:
        return "Play = Yes"
    conditions = []
    for attribute, value in rule:
        conditions.append(
            f"{attribute} = {value}"
        )
    return "IF " + " AND ".join(conditions) + " THEN Play = Yes"

In [35]:
def learn_one_rule(df, target, positive_value, negative_value, features):
    rule = []
    while True:
        covered = covered_examples(df, rule)
        positives = covered[
            covered[target] == positive_value
        ]
        negatives = covered[
            covered[target] == negative_value
        ]
        if len(negatives) == 0:
            if len(positives) > 0:
                return rule
            return None
        p0 = len(positives)
        n0 = len(negatives)
        candidates = generate_candidate_literals(
            df,
            rule,
            features
        )
        if len(candidates) == 0:
            return None
        best_literal = None
        best_gain = float("-inf")
        for literal in candidates:
            new_rule = rule + [literal]
            new_covered = covered_examples(
                df,
                new_rule
            )
            p1 = len(
                new_covered[
                    new_covered[target] == positive_value
                ]
            )
            n1 = len(
                new_covered[
                    new_covered[target] == negative_value
                ]
            )
            gain = foil_gain(
                p0,
                n0,
                p1,
                n1
            )
            print(
                f"    Candidate: "
                f"{literal_to_string(literal):35s} "
                f"Gain = {gain:.4f}"
            )
            if gain > best_gain:
                best_gain = gain
                best_literal = literal
        if best_literal is None:
            return None
        rule.append(best_literal)
        print(
            f"\n    Selected: "
            f"{literal_to_string(best_literal)}"
        )
        print(
            f"    Current rule: {rule_to_string(rule)}"
        )
        print()
        if len(rule) >= len(features):
            break
    return rule

In [36]:
def foil(df, target, positive_value, negative_value, features):
    positives_remaining = df[
        df[target] == positive_value
    ].copy()
    negatives = df[
        df[target] == negative_value
    ].copy()
    learned_rules = []
    print("\n==============================")
    print("         FOIL START")
    print("==============================\n")
    while len(positives_remaining) > 0:
        print(
            f"Positive examples remaining: "
            f"{len(positives_remaining)}"
        )
        rule = learn_one_rule(
            pd.concat(
                [positives_remaining, negatives]
            ),
            target,
            positive_value,
            negative_value,
            features
        )
        if rule is None:
            print("Could not find another rule.")
            break
        learned_rules.append(rule)
        print("\nLearned Rule:")
        print("  ", rule_to_string(rule))
        covered_positive = covered_examples(
            positives_remaining,
            rule
        )
        print(
            f"Covered positive examples: "
            f"{len(covered_positive)}"
        )
        positives_remaining = positives_remaining.drop(
            covered_positive.index
        )
        print()
    return learned_rules

In [37]:
def predict(example, rules):
    for rule in rules:
        if satisfies(example, rule):
            return "Yes"
    return "No"
if __name__ == "__main__":
    df = pd.read_csv("small_dataset.csv")
    target = "Play"
    features = [
        "Weather",
        "Temperature",
        "Humidity",
        "Wind"
    ]
    rules = foil(
        df=df,
        target=target,
        positive_value="Yes",
        negative_value="No",
        features=features
    )
    print("\n==============================")
    print("       FINAL FOIL RULES")
    print("==============================\n")

    for i, rule in enumerate(rules, 1):

        print(
            f"Rule {i}: "
            f"{rule_to_string(rule)}"
        )
    print("\n==============================")
    print("          PREDICTIONS")
    print("==============================\n")

    for _, row in df.iterrows():

        prediction = predict(row, rules)

        print(
            f"Actual: {row[target]:3s} | "
            f"Predicted: {prediction}"
        )


         FOIL START

Positive examples remaining: 9
    Candidate: Weather = Overcast                  Gain = 2.5497
    Candidate: Weather = Rain                      Gain = -0.2986
    Candidate: Weather = Sunny                     Gain = -1.3690
    Candidate: Temperature = Hot                   Gain = -0.7251
    Candidate: Temperature = Mild                  Gain = 0.2099
    Candidate: Temperature = Cool                  Gain = 0.6672
    Candidate: Humidity = High                     Gain = -1.7549
    Candidate: Humidity = Normal                   Gain = 2.4902
    Candidate: Wind = Weak                         Gain = 1.3344
    Candidate: Wind = Strong                       Gain = -1.0877

    Selected: Weather = Overcast
    Current rule: IF Weather = Overcast THEN Play = Yes


Learned Rule:
   IF Weather = Overcast THEN Play = Yes
Covered positive examples: 4

Positive examples remaining: 5
    Candidate: Weather = Rain                      Gain = 0.7891
    Candidate: Weat